In [59]:
from_date = 20241107
to_date = 20241107

from datetime import datetime

# Convert dates to UNIX timestamps
from_date_unix = int(datetime.strptime(str(from_date) + ' 00:00:00', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours
to_date_unix = int(datetime.strptime(str(to_date) + ' 23:59:59', '%Y%m%d %H:%M:%S').timestamp()) + 25200 # Add 7 hours
print(from_date_unix)
print(to_date_unix)

1730937600
1731023999


In [74]:
from helper.config import load_config
import psycopg2

import pandas as pd
import numpy as np

config = load_config()
with psycopg2.connect(**config) as conn:
    with conn.cursor() as cur:
            cur.execute(f"""
                    select 
                        a.start_time
                        ,a.end_time
                        ,c.lat as start_lat
                        ,c.lon as start_lon
                        ,d.lat as end_lat
                        ,d.lon as end_lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name as start_location_name
                        ,c.category as start_category
                        ,d.location_name as end_location_name
                        ,d.category as end_category
                        ,case 
                            when json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                            ) filter (where b.txtime is not null or b.lat is not null or b.lon is not null) is null 
                            then null
                            else json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                            )
                        end as path 
                    from gmap.fact_activity a
                    left join gmap.fact_timelinepath b
                    on b.txtime between a.start_time and a.end_time 
                    left join gmap.dim_location c
                    on a.start_location_id = c.location_id
                    left join gmap.dim_location d
                    on a.end_location_id = d.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    group by 
                        a.start_time
                        ,a.end_time
                        ,c.lat
                        ,c.lon
                        ,d.lat
                        ,d.lon
                        ,a.vehicle_type
                        ,a.distance_meters
                        ,c.location_name
                        ,c.category
                        ,d.location_name
                        ,d.category
                        """)
            activity = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

            cur.execute(f"""
                    select 
                        a.start_time 
                        ,a.end_time 
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        ,case 
                            when json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                            ) filter (where b.txtime is not null or b.lat is not null or b.lon is not null) is null 
                            then null
                            else json_agg(
                                json_build_object('txtime', b.txtime, 'lat', b.lat, 'lon', b.lon)
                            )
                        end as path
                    from gmap.fact_visit a
                    left join gmap.fact_timelinepath b
                    on b.txtime between a.start_time and a.end_time 
                    left join gmap.dim_location c
                    on a.location_id = c.location_id
                    where a.start_time between {from_date_unix} AND {to_date_unix}
                    group by 
                        a.start_time 
                        ,a.end_time 
                        ,c.lat 
                        ,c.lon
                        ,c.location_name
                        ,c.category
                        """)
            visit = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description]) 

# add info to activity
activity['path']
activity['path'] = activity['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x} if x is not None else None)
activity['color'] = activity['vehicle_type'].map({
    'in bus': 'red',
    'in passenger vehicle': 'blue',
    'walking': 'green',
    'motorcycling': 'yellow',
    'unknown': 'black'
})
activity['start_time_human'] = pd.to_datetime(activity['start_time'], unit='s')
activity['end_time_human'] = pd.to_datetime(activity['end_time'], unit='s')

# add info to visit
visit['path'] = visit['path'].apply(lambda x: {item['txtime']: [item['lat'], item['lon']] for item in x} if x is not None else None)
visit['duration'] = visit['end_time'] - visit['start_time']
visit['duration'] = visit['duration'].apply(lambda x: '{} hours {} minutes'.format(int(divmod(x, 60*60)[0]), int(divmod(divmod(x, 60*60)[1], 60)[0])))
visit['start_time_human'] = pd.to_datetime(visit['start_time'], unit='s')
visit['end_time_human'] = pd.to_datetime(visit['end_time'], unit='s')

# Get all coordinates for map zoom display
visit_path_coordinates = sum(visit['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
visit_coordinates = visit[['lat', 'lon']].apply(lambda x: [float(x['lat']), float(x['lon'])], axis=1).to_list()
activity_path_coordinates = sum(activity['path'].apply(lambda x: list(x.values() if isinstance(x, dict) else [])),[])
activity_start_coordinates = activity[['start_lat', 'start_lon']].apply(lambda x: [float(x['start_lat']), float(x['start_lon'])], axis=1).to_list()
activity_end_coordinates = activity[['end_lat', 'end_lon']].apply(lambda x: [float(x['end_lat']), float(x['end_lon'])], axis=1).to_list()
coordinates = sum([visit_path_coordinates, visit_coordinates, activity_path_coordinates, activity_start_coordinates, activity_end_coordinates], [])
min_lat = min((coord[0] for coord in coordinates if coord[0] is not None), default=None)
max_lat = max((coord[0] for coord in coordinates if coord[0] is not None), default=None)
min_lon = min((coord[1] for coord in coordinates if coord[1] is not None), default=None)
max_lon = max((coord[1] for coord in coordinates if coord[1] is not None), default=None)

pd.DataFrame(
      np.concatenate([
      activity[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_name', 'end_location_name', 'vehicle_type']].to_numpy(), 
      visit[['start_time_human', 'end_time_human', 'start_time', 'end_time', 'location_name', 'location_name', 'location_name']].to_numpy()
      ])
      , columns=['start_time_human', 'end_time_human', 'start_time', 'end_time', 'start_location_name', 'end_location_name', 'vehicle_type']
      ).sort_values('start_time')


,start_time_human,end_time_human,start_time,end_time,start_location_name,end_location_name,vehicle_type
1,2024-11-07 08:03:23,2024-11-07 08:10:19,1730966603,1730967019,Home,807 Giải Phóng - giáp Kim Đồng,walking
0,2024-11-07 08:10:20,2024-11-07 08:45:28,1730967020,1730969128,807 Giải Phóng - giáp Kim Đồng,162 Khuất Duy Tiến - giáp Lê Văn Lương,in bus
7,2024-11-07 08:45:29,2024-11-07 11:42:00,1730969129,1730979720,Work,Work,Work
4,2024-11-07 11:42:01,2024-11-07 11:47:03,1730979721,1730980023,Work,Bún Bò Huế Ngự Uyển - Nguỵ Như Kon Tum,walking
10,2024-11-07 11:47:04,2024-11-07 12:19:49,1730980024,1730981989,Bún Bò Huế Ngự Uyển - Nguỵ Như Kon Tum,Bún Bò Huế Ngự Uyển - Nguỵ Như Kon Tum,Bún Bò Huế Ngự Uyển - Nguỵ Như Kon Tum
2,2024-11-07 12:19:50,2024-11-07 12:28:20,1730981990,1730982500,Bún Bò Huế Ngự Uyển - Nguỵ Như Kon Tum,Work,walking
9,2024-11-07 12:28:21,2024-11-07 18:18:21,1730982501,1731003501,Work,Work,Work
3,2024-11-07 18:18:22,2024-11-07 18:45:02,1731003502,1731005102,39 Khuất Duy Tiến - giáp Tổ Hữu,Quận ủy Thanh Xuân - 9 Khuất Duy Tiến,walking
6,2024-11-07 18:45:03,2024-11-07 19:17:22,1731005103,1731007042,Quận ủy Thanh Xuân - 9 Khuất Duy Tiến,Đối Diện 807 Giải Phóng - giáp Định Công,in bus
5,2024-11-07 19:17:23,2024-11-07 19:24:58,1731007043,1731007498,Đối Diện 807 Giải Phóng - giáp Định Công,Home,walking


In [73]:
import folium
from folium.plugins import MarkerCluster, AntPath

f = folium.Figure(width=1000, height=600)
m = folium.Map(
                location=[(min_lat + max_lat)/2, (min_lon + max_lon)/2],
                zoom_start=13, 
                control_scale=True,
                tiles="cartodbpositron",
               ).add_to(f)

# # if the points are too close to each other, cluster them, create a cluster overlay with MarkerCluster
marker_cluster = MarkerCluster().add_to(m)

for _, item in activity.iterrows():
    # print(item['path'])
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Transportation: '\
                +item['vehicle_type']\
                +'</p>'
    if isinstance(item['path'], dict):
        AntPath(
            locations=list(item['path'].values()), 
            color=item['color'],
            delay=800,
            weight=5,
            opacity=0.5,
            # reverse="True", 
            dash_array=[10, 20],
            tooltip=tooltip_txt
        ).add_to(m)

    folium.Marker(
                    location = (item['start_lat'], item['start_lon']),
                    icon = folium.Icon(icon='play', prefix='glyphicon', color='orange'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
    
    folium.Marker(
                    location = (item['end_lat'], item['end_lon']),
                    icon = folium.Icon(icon='stop', prefix='glyphicon', color='blue'),
                    tooltip=tooltip_txt
                ).add_to(marker_cluster)    
        
for _, item in visit.iterrows():
    if isinstance(item['path'], dict):
        folium.PolyLine(
            list(item['path'].values()), 
            color='black',
            weight=2,
            opacity=0.5,
            # dash_array='5, 5'
            ).add_to(m)
    
    tooltip_txt = '<p>'\
                +'From: '\
                +pd.to_datetime(item['start_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> To: '\
                +pd.to_datetime(item['end_time'], unit = 's').strftime('%Y-%m-%d %H:%M:%S')\
                +'<br> Duration: '\
                +item['duration']\
                +'</p>'
    
    folium.Marker(
                    location = (item['lat'], item['lon']),
                    icon = folium.Icon(icon='ok', prefix='glyphicon', color='green'),
                    tooltip = tooltip_txt
                ).add_to(marker_cluster)    
    
m.fit_bounds(m.get_bounds())

m